# 03 — Temporal Prediction

This notebook demonstrates wavefront prediction for servo-lag
compensation:

1. Load Zernike time series from the dataset
2. Train an LSTM for a few epochs (quick demo)
3. Plot prediction accuracy per Zernike mode
4. Run `ClosedLoopSimulator` with and without prediction
5. Plot Strehl ratio time series: open loop / closed loop / predictive AO
6. Run `TurbulenceParameterEstimator`, compare estimated vs true r0 and
   Greenwood frequency
7. Animate the DM surface and residual wavefront over 20 frames


In [1]:
import sys
sys.path.insert(0, "..")

import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import yaml

from sim.dataset_gen import generate_dataset, load_dataset
from sim.turbulence import build_atmosphere_from_config
from sim.shwfs import SHWFSSensor
from reconstruction.zernike import zernike_basis
from reconstruction.classical import ModalReconstructor
from actuator.dm_command import DMController
from temporal.lstm_model import ZernikeTimeSeries
from temporal.train_temporal import prepare_sequences, train_temporal_model, evaluate_prediction_accuracy
from temporal.predictor import WavefrontPredictor, ClosedLoopSimulator
from temporal.turbulence_param import TurbulenceParameterEstimator
from viz.plot_utils import plot_strehl_timeseries, animate_closed_loop, plot_phase_map
from sim.phase_screen import get_aperture_mask, apply_aperture_mask

with open("../config.yaml") as f:
    config = yaml.safe_load(f)

config["sim"]["n_frames"] = 600
config["sim"]["grid_size"] = 64
config["temporal"]["n_epochs"] = 10
config["temporal"]["sequence_length"] = 20
config["temporal"]["batch_size"] = 16

sim_cfg = config["sim"]
N = sim_cfg["grid_size"]
n_zernike = sim_cfg["n_zernike"]


## 1. Load Zernike time series


In [2]:
os.makedirs("../data", exist_ok=True)
os.makedirs("../models", exist_ok=True)

dataset_path = "../data/dataset_temporal.h5"
if not os.path.exists(dataset_path):
    generate_dataset(config, n_frames=sim_cfg["n_frames"], output_path=dataset_path, seed=7)

data = load_dataset(dataset_path)
zernike_data = data["zernike_coeffs"]
print("zernike_coeffs shape:", zernike_data.shape)

fig, ax = plt.subplots(figsize=(8, 4))
for j in [1, 2, 3]:  # tip, tilt, defocus (0-indexed)
    ax.plot(zernike_data[:200, j], label=f"mode {j+1}")
ax.set_xlabel("frame")
ax.set_ylabel("coefficient (rad)")
ax.set_title("Zernike coefficient time series (first 200 frames)")
ax.legend()
plt.show()


Generating dataset:   0%|          | 0/600 [00:00<?, ?it/s]

Generating dataset:   1%|          | 5/600 [00:00<00:14, 41.46it/s]

Generating dataset:   2%|▏         | 10/600 [00:00<00:14, 41.80it/s]

Generating dataset:   2%|▎         | 15/600 [00:00<00:13, 42.09it/s]

Generating dataset:   3%|▎         | 20/600 [00:00<00:13, 42.16it/s]

...

Generating dataset:  98%|█████████▊| 587/600 [00:14<00:00, 40.98it/s]

Generating dataset:  99%|█████████▊| 592/600 [00:14<00:00, 40.25it/s]

Generating dataset: 100%|█████████▉| 597/600 [00:14<00:00, 40.36it/s]

Generating dataset: 100%|██████████| 600/600 [00:14<00:00, 40.66it/s]

zernike_coeffs shape: (600, 36)


## 2. Train LSTM (quick demo)


In [3]:
lstm_save_path = "../models/temporal_model_demo.pt"
history = train_temporal_model(config, dataset_path, lstm_save_path)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(history["train_loss"], label="train loss")
ax.plot(history["val_loss"], label="val loss")
ax.set_xlabel("epoch")
ax.set_ylabel("MSE loss")
ax.set_title("LSTM training curve")
ax.legend()
plt.show()


Training temporal model:   0%|          | 0/10 [00:00<?, ?it/s]

Training temporal model:   0%|          | 0/10 [00:00<?, ?it/s, train_loss=0.032948, val_loss=0.011844]

Training temporal model:  10%|█         | 1/10 [00:00<00:03,  2.98it/s, train_loss=0.032948, val_loss=0.011844]

Training temporal model:  10%|█         | 1/10 [00:00<00:03,  2.98it/s, train_loss=0.007483, val_loss=0.003250]

Training temporal model:  20%|██        | 2/10 [00:00<00:02,  3.59it/s, train_loss=0.007483, val_loss=0.003250]

...

Training temporal model:  80%|████████  | 8/10 [00:02<00:00,  4.20it/s, train_loss=0.000727, val_loss=0.000622]

Training temporal model:  90%|█████████ | 9/10 [00:02<00:00,  4.28it/s, train_loss=0.000727, val_loss=0.000622]

Training temporal model:  90%|█████████ | 9/10 [00:02<00:00,  4.28it/s, train_loss=0.000675, val_loss=0.000605]

Training temporal model: 100%|██████████| 10/10 [00:02<00:00,  4.32it/s, train_loss=0.000675, val_loss=0.000605]

Training temporal model: 100%|██████████| 10/10 [00:02<00:00,  4.08it/s, train_loss=0.000675, val_loss=0.000605]

## 3. Prediction accuracy per Zernike mode


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ZernikeTimeSeries(
    input_size=n_zernike,
    hidden_size=config["temporal"]["hidden_size"],
    n_layers=config["temporal"]["n_layers"],
    output_size=n_zernike,
).to(device)
checkpoint = torch.load(lstm_save_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

seq_len = config["temporal"]["sequence_length"]
horizon = config["temporal"]["predict_horizon"]
X, y = prepare_sequences(zernike_data, seq_len, horizon)

model_rmse, baseline_rmse = evaluate_prediction_accuracy(model, (X, y), sim_cfg["dt_s"], config)

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(1, n_zernike + 1)
ax.plot(x, model_rmse, "o-", label="LSTM prediction")
ax.plot(x, baseline_rmse, "s--", label="persistence baseline")
ax.set_xlabel("Noll index j")
ax.set_ylabel("RMSE (rad)")
ax.set_title("Per-mode prediction RMSE: LSTM vs persistence")
ax.legend()
plt.show()

improvement = (baseline_rmse - model_rmse) / (baseline_rmse + 1e-12)
print(f"Mean improvement over persistence: {improvement.mean()*100:.1f}%")


Mean improvement over persistence: -145.3%


## 4 & 5. Closed-loop simulation with and without prediction; Strehl time series


In [5]:
pixel_scale = sim_cfg["aperture_diameter_m"] / N
atmosphere = build_atmosphere_from_config(config, N=N, pixel_scale=pixel_scale, seed=21)

sensor = SHWFSSensor(
    n_subapertures=sim_cfg["n_subapertures"],
    pixels_per_subaperture=sim_cfg["detector_pixels_per_subaperture"],
    focal_length=sim_cfg["focal_length_m"],
    pitch=sim_cfg["mla_pitch_m"],
    wavelength=sim_cfg["wavelength_m"],
)
basis = zernike_basis(n_zernike, N)
modal_recon = ModalReconstructor(sensor, basis, n_zernike, config["reconstruction"]["svd_condition_number"])
dm = DMController(config)

predictor = WavefrontPredictor(model, seq_len, device)
closed_loop = ClosedLoopSimulator(atmosphere, sensor, dm, modal_recon, predictor, config)

n_frames = 100
results = closed_loop.run(n_frames=n_frames, use_prediction=True)

t = np.arange(n_frames) * sim_cfg["dt_s"]
strehl_open = np.exp(-(results["rms_open_loop"] ** 2))

fig, ax = plt.subplots(figsize=(8, 4))
plot_strehl_timeseries(t, strehl_open, results["strehl_no_pred"], results["strehl_with_pred"], ax=ax)
plt.show()

print(f"Mean Strehl (open loop):     {strehl_open.mean():.4f}")
print(f"Mean Strehl (closed loop):   {results['strehl_no_pred'].mean():.4f}")
print(f"Mean Strehl (predictive AO): {results['strehl_with_pred'].mean():.4f}")


Mean Strehl (open loop):     0.0000
Mean Strehl (closed loop):   0.8914
Mean Strehl (predictive AO): 0.1596


## 6. Turbulence parameter estimation


In [6]:
estimator = TurbulenceParameterEstimator(
    wavelength=sim_cfg["wavelength_m"], D=sim_cfg["aperture_diameter_m"],
    dt=sim_cfg["dt_s"], n_subapertures=sim_cfg["n_subapertures"],
)
report = estimator.fit(zernike_data)

print("Estimated turbulence parameters:")
for k, v in report.items():
    print(f"  {k}: {v:.5f}" if isinstance(v, float) else f"  {k}: {v}")

print(f"\nTrue r0:           {config['turbulence']['r0_m']:.4f} m")


Estimated turbulence parameters:
  r0_m: 0.41302
  r0_m_std: 0.01117
  greenwood_freq_hz: 43.90882
  greenwood_freq_hz_std: 75.14369
  wind_speed_ms: 10.00000
  wind_direction_deg: 0.00052

True r0:           0.1500 m


In [7]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(["estimated", "true"], [report["r0_m"], config["turbulence"]["r0_m"]],
            yerr=[report["r0_m_std"], 0])
axes[0].set_title("Fried parameter r0 (m)")

axes[1].bar(["estimated"], [report["greenwood_freq_hz"]], yerr=[report["greenwood_freq_hz_std"]])
axes[1].set_title("Greenwood frequency (Hz)")

plt.tight_layout()
plt.show()


## 7. Animate DM surface and residual wavefront over 20 frames


In [8]:
mask = get_aperture_mask(N, "circular")
wavelength_factor = sim_cfg["wavelength_m"] / (2 * np.pi)

atmosphere2 = build_atmosphere_from_config(config, N=N, pixel_scale=pixel_scale, seed=22)
dm.reset_integrator()

n_anim_frames = 20
phase_seq = np.zeros((n_anim_frames, N, N))
dm_seq = np.zeros((n_anim_frames, N, N))
residual_seq = np.zeros((n_anim_frames, N, N))

for k in range(n_anim_frames):
    atmosphere2.evolve(sim_cfg["dt_s"])
    phase_rad = atmosphere2.get_integrated_phase_radians()
    phase_masked = apply_aperture_mask(phase_rad, "circular")
    phase_m = phase_masked * wavelength_factor

    commands, residual = dm.closed_loop_step(phase_m, mask, gain=0.5)
    dm_surface = dm.commands_to_surface(commands)

    phase_seq[k] = phase_m
    dm_seq[k] = dm_surface
    residual_seq[k] = residual

os.makedirs("../results", exist_ok=True)
output_path = animate_closed_loop(phase_seq, dm_seq, residual_seq, "../results/closed_loop_animation.mp4")
print(f"Animation saved to: {output_path}")


Animation saved to: ../results/closed_loop_animation.mp4


In [9]:
# Display first and last frames as static comparison
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for col, (title, seq) in enumerate(zip(["Phase", "DM surface", "Residual"], [phase_seq, dm_seq, residual_seq])):
    plot_phase_map(seq[0], f"{title} (t=0)", mask=mask, ax=axes[0, col], units="m")
    plot_phase_map(seq[-1], f"{title} (t={n_anim_frames-1})", mask=mask, ax=axes[1, col], units="m")
plt.tight_layout()
plt.show()

rms_initial = np.sqrt(np.mean(residual_seq[0][mask] ** 2))
rms_final = np.sqrt(np.mean(residual_seq[-1][mask] ** 2))
print(f"Residual RMS at t=0:  {rms_initial*1e9:.2f} nm")
print(f"Residual RMS at t={n_anim_frames-1}: {rms_final*1e9:.2f} nm")


Residual RMS at t=0:  209.27 nm
Residual RMS at t=19: 24.05 nm
